<a href="https://colab.research.google.com/github/minyi-k03/LargeLanguageModel/blob/Project-Based-Learning(PBL)/LLM_Guardrails_%EA%B8%B0%EC%B4%88_%EA%B5%AC%ED%98%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Guardrails 기초 구현 예제

## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )

# Reference : https://cookbook.openai.com/examples/how_to_use_guardrails

## openai-moderation-api-evaluation Dataset : https://huggingface.co/datasets/mmathys/openai-moderation-api-evaluation

In [ ]:
!pip install openai

In [ ]:
!pip show openai

## OpenAI API Key 설정

In [ ]:
import os
from openai import OpenAI

# 3. API 키 설정
os.environ["OPENAI_API_KEY"] = "Input Your Key"

# 4. 클라이언트 생성 (이제 이 client 객체를 통해 통신합니다)
client = OpenAI()

# 5. 모델 설정
GPT_MODEL = 'gpt-4o-mini'


# 1. Input guardrails

In [ ]:
#system_prompt = "You are a helpful assistant."
system_prompt = "당신은 도움이 되는 어시스턴트입니다."

#bad_request = "I want to talk about horses"
bad_request = "나는 말에 대해 이야기하고 싶어요."

#good_request = "What are the best breeds of dog for people that like cats?"
good_request = "고양이를 좋아하는 사람들에게 가장 잘 맞는 개 품종은 무엇인가요?"

In [ ]:
import asyncio


async def get_chat_response(user_request):
    print("Getting LLM response")
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_request},
    ]
    response = client.chat.completions.create(
        model=GPT_MODEL, messages=messages, temperature=0.5
    )
    print("Got LLM response")

    return response.choices[0].message.content


async def topical_guardrail(user_request):
    print("Checking topical guardrail")
    messages = [
        {
            "role": "system",
            #"content": "Your role is to assess whether the user question is allowed or not. The allowed topics are cats and dogs. If the topic is allowed, say 'allowed' otherwise say 'not_allowed'",
            "content": "당신의 역할은 사용자의 질문이 허용되는지 여부를 판단하는 것입니다. 허용된 주제는 고양이와 개입니다. 주제가 허용된다면 'allowed'라고 말하고, 그렇지 않다면 'not_allowed'라고 말하세요.",
        },
        {"role": "user", "content": user_request},
    ]
    response = client.chat.completions.create(
        model=GPT_MODEL, messages=messages, temperature=0
    )

    print("Got guardrail response")
    return response.choices[0].message.content


async def execute_chat_with_guardrail(user_request):
    topical_guardrail_task = asyncio.create_task(topical_guardrail(user_request))
    chat_task = asyncio.create_task(get_chat_response(user_request))

    while True:
        done, _ = await asyncio.wait(
            [topical_guardrail_task, chat_task], return_when=asyncio.FIRST_COMPLETED
        )
        if topical_guardrail_task in done:
            guardrail_response = topical_guardrail_task.result()
            if guardrail_response == "not_allowed":
                chat_task.cancel()
                print("Topical guardrail triggered")
                #return "I can only talk about cats and dogs, the best animals that ever lived."
                return "나는 오직 고양이와 개에 대해서만 이야기할 수 있어요. 그들은 지금까지 존재한 최고의 동물들이에요."
            elif chat_task in done:
                chat_response = chat_task.result()
                return chat_response
        else:
            await asyncio.sleep(0.1)  # sleep for a bit before checking the tasks again

In [ ]:
#주제에 맞는 질문을 한 경우
# Call the main function with the good request - this should go through
response = await execute_chat_with_guardrail(good_request)
print(response)

In [ ]:
#주제에 벗어난 질문을 한 경우
# Call the main function with the bad request - this should get blocked
response = await execute_chat_with_guardrail(bad_request)
print(response)

# 2. Output guardrails

In [ ]:
#domain = "animal breed recommendation"
#주제 설정
domain = "동물 품종 추천"

# animal_advice_criteria = """
# Assess the presence of explicit recommendation of cat or dog breeds in the content.
# The content should contain only general advice about cats and dogs, not specific breeds to purchase."""
animal_advice_criteria = """
콘텐츠에 고양이나 개 품종에 대한 명시적인 추천이 포함되어 있는지 평가하세요.
콘텐츠에는 특정 품종을 구입하라는 내용이 아니라, 고양이와 개에 대한 일반적인 조언만 포함되어야 합니다."""

# animal_advice_steps = """
# 1. Read the content and the criteria carefully.
# 2. Assess how much explicit recommendation of cat or dog breeds is contained in the content.
# 3. Assign an animal advice score from 1 to 5, with 1 being no explicit cat or dog breed advice, and 5 being multiple named cat or dog breeds.
# """
animal_advice_steps = """
1. 콘텐츠와 기준을 주의 깊게 읽으세요.
2. 콘텐츠에 고양이나 개 품종에 대한 명시적인 추천이 얼마나 포함되어 있는지 평가하세요.
3. 고양이나 개 품종에 대한 조언의 명시적 정도에 따라 1부터 5까지의 점수를 부여하세요. 1점: 품종에 대한 명시적인 조언이 전혀 없음, 5점: 여러 개의 고양이 또는 개 품종이 구체적으로 언급되어 있음
"""

# moderation_system_prompt = """
# You are a moderation assistant. Your role is to detect content about {domain} in the text provided, and mark the severity of that content.

# ## {domain}

# ### Criteria

# {scoring_criteria}

# ### Instructions

# {scoring_steps}

# ### Content

# {content}

# ### Evaluation (score only!)
# """
moderation_system_prompt = """
당신은 모더레이션 어시스턴트입니다. 당신의 역할은 제공된 텍스트에서 {domain}에 대한 내용을 감지하고, 해당 내용의 심각도를 평가하는 것입니다.

## {domain}

### Criteria

{scoring_criteria}

### Instructions

{scoring_steps}

### Content

{content}

### Evaluation (score only!)
"""

In [ ]:
async def moderation_guardrail(chat_response):
    print("Checking moderation guardrail")
    mod_messages = [
        {"role": "user", "content": moderation_system_prompt.format(
            domain=domain,
            scoring_criteria=animal_advice_criteria,
            scoring_steps=animal_advice_steps,
            content=chat_response
        )},
    ]
    response = client.chat.completions.create(
        model=GPT_MODEL, messages=mod_messages, temperature=0
    )
    print("Got moderation response")
    return response.choices[0].message.content


async def execute_all_guardrails(user_request):
    topical_guardrail_task = asyncio.create_task(topical_guardrail(user_request))
    chat_task = asyncio.create_task(get_chat_response(user_request))

    while True:
        done, _ = await asyncio.wait(
            [topical_guardrail_task, chat_task], return_when=asyncio.FIRST_COMPLETED
        )
        if topical_guardrail_task in done:
            guardrail_response = topical_guardrail_task.result()
            if guardrail_response == "not_allowed":
                chat_task.cancel()
                print("Topical guardrail triggered")
                #return "I can only talk about cats and dogs, the best animals that ever lived."
                return "나는 오직 고양이와 개에 대해서만 이야기할 수 있어요. 그들은 지금까지 존재한 최고의 동물들이에요."
            elif chat_task in done:
                chat_response = chat_task.result()
                moderation_response = await moderation_guardrail(chat_response)

                if int(moderation_response) >= 3:
                    print(f"Moderation guardrail flagged with a score of {int(moderation_response)}")
                    #return "Sorry, we're not permitted to give animal breed advice. I can help you with any general queries you might have."
                    return "죄송하지만, 동물 품종에 대한 조언은 제공해 드릴 수 없습니다. 일반적인 질문이라면 언제든지 도와드릴 수 있어요."
                else:
                    print('Passed moderation')
                    return chat_response
        else:
            await asyncio.sleep(0.1)  # sleep for a bit before checking the tasks again

In [ ]:
# Adding a request that should pass both our topical guardrail and our moderation guardrail
# great_request = 'What is some advice you can give to a new dog owner?'
great_request = '새로 개를 키우기 시작한 사람에게 줄 수 있는 조언은 무엇인가요?'

In [ ]:
tests = [good_request, bad_request, great_request]

for test in tests:
    result = await execute_all_guardrails(test)
    print(result)
    print('\n\n')